# Evaluation: Data Understanding Node
This notebook evaluates `data_understanding` predictions against benchmark labels with a **multi-label** setup for three variables:
- `data_category`
- `data_format`
- `data_characteristics`

In [1]:
# Setup imports and project paths
import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer

# Resolve project root robustly from notebook location
ROOT = Path.cwd().resolve()
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv"
RESULTS_FOLDER = PILOT / "pilot_study_results"

print(f"Project root: {ROOT}")
print(f"Benchmark file exists: {BENCHMARK_FILE.exists()}")
print(f"Results folder exists: {RESULTS_FOLDER.exists()}")

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True


In [2]:
# Load and clean benchmark labels
benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()

comment_cols = [
    c for c in benchmark_df.columns
    if "Any Comments" in c or "Any comments" in c
]
benchmark_df_clean = benchmark_df.drop(columns=comment_cols, errors="ignore")

required_cols = [
    "Paper ID",
    "Gatekeeper",
    "data_category",
    "data_format",
    "data_characteristics"
]
missing = [c for c in required_cols if c not in benchmark_df_clean.columns]
if missing:
    raise ValueError(f"Missing required benchmark columns: {missing}")

print(f"Loaded benchmark rows: {len(benchmark_df_clean)}")
print(benchmark_df_clean[required_cols].head())

Loaded benchmark rows: 10
    Paper ID Gatekeeper         data_category data_format  \
0   2011 - 1    Exclude                   NaN         NaN   
1   2018 - 8    Include  Survey / Statistical  Structured   
2  2018 - 12    Exclude                   NaN         NaN   
3  2018 - 26    Exclude                   NaN         NaN   
4  2019 - 33    Include                   UGC  Structured   

       data_characteristics  
0                       NaN  
1  Temporal;Spatial;Textual  
2                       NaN  
3                       NaN  
4         Temporal; Textual  


In [3]:
# Helper functions and expected label spaces
DIMENSIONS = ["data_category", "data_format", "data_characteristics"]

LABEL_SPACES = {
    "data_category": ["UGC", "Device", "Transaction", "Survey/Statistical"],
    "data_format": ["Structured", "Semi-Structured", "Unstructured"],
    "data_characteristics": ["Temporal", "Spatial", "Textual", "Visual", "Networked"],
}

SYNONYM_MAP = {
    "data_category": {
        "ugc": "UGC",
        "device": "Device",
        "transaction": "Transaction",
        "survey/statistical": "Survey/Statistical",
        "survey / statistical": "Survey/Statistical",
    },
    "data_format": {
        "structured": "Structured",
        "semi-structured": "Semi-Structured",
        "semistructured": "Semi-Structured",
        "unstructured": "Unstructured",
    },
    "data_characteristics": {
        "temporal": "Temporal",
        "spatial": "Spatial",
        "textual": "Textual",
        "visual": "Visual",
        "networked": "Networked",
    },
}

def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def clean_label_token(token):
    t = token.strip().lower()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" / ", "/")
    return t

def parse_multilabel(value, dimension):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        raw_items = value
    else:
        text = str(value).strip()
        if not text:
            return []
        # Split by semicolon first (benchmark convention), fallback to comma
        raw_items = [x for part in text.split(";") for x in part.split(",")]

    mapped = []
    for item in raw_items:
        token = clean_label_token(str(item))
        canonical = SYNONYM_MAP[dimension].get(token)
        if canonical:
            mapped.append(canonical)

    # Remove duplicates while preserving order
    seen = set()
    out = []
    for label in mapped:
        if label not in seen:
            seen.add(label)
            out.append(label)
    return out

def get_nested(data, path, default=None):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

def load_agent_results(results_folder: Path):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob("*/")):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / "aggregated" / "results.json"
        if not results_file.exists():
            continue
        with open(results_file, "r", encoding="utf-8") as f:
            agent_data[paper_dir.name.strip().lower()] = json.load(f)
    return agent_data

agent_data = load_agent_results(RESULTS_FOLDER)
print(f"Loaded aggregated agent outputs for {len(agent_data)} papers")

Loaded aggregated agent outputs for 10 papers


In [4]:
# Build comparison table for data_understanding (only Gatekeeper == include)
comparison_rows = []
included_rows = 0
skipped_not_included = 0
missing_agent_results = []

for _, row in benchmark_df_clean.iterrows():
    gatekeeper_label = normalize_text(row.get("Gatekeeper"))
    if (gatekeeper_label or "").strip().lower() != "include":
        skipped_not_included += 1
        continue

    included_rows += 1
    paper_id = normalize_paper_id(row.get("Paper ID"))
    if not paper_id:
        continue

    if paper_id not in agent_data:
        missing_agent_results.append(paper_id)
        continue

    results = agent_data[paper_id]
    data_understanding = get_nested(results, "dsrp_outputs.data_understanding", default={})

    record = {"Paper ID": paper_id}
    row_match_flags = []

    for dim in DIMENSIONS:
        gt_labels = parse_multilabel(row.get(dim), dim)
        pred_labels = parse_multilabel(data_understanding.get(dim), dim)

        gt_set = sorted(set(gt_labels))
        pred_set = sorted(set(pred_labels))
        dim_match = gt_set == pred_set
        row_match_flags.append(dim_match)

        record[f"GT_{dim}"] = gt_set
        record[f"Agent_{dim}"] = pred_set
        record[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

    record["Match_All_3"] = "MATCH" if all(row_match_flags) else "MISMATCH"
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)

print(f"Benchmark rows with Gatekeeper=include: {included_rows}")
print(f"Skipped (Gatekeeper!=include): {skipped_not_included}")
print(f"Rows with agent results: {len(comparison_df)}")
if missing_agent_results:
    print(f"Missing agent outputs for: {', '.join(missing_agent_results)}")

display_cols = ["Paper ID", "Match_data_category", "Match_data_format", "Match_data_characteristics", "Match_All_3"]
if len(comparison_df):
    print(comparison_df[display_cols].to_string(index=False))
else:
    print("No comparable rows found.")

Benchmark rows with Gatekeeper=include: 7
Skipped (Gatekeeper!=include): 3
Rows with agent results: 7
 Paper ID Match_data_category Match_data_format Match_data_characteristics Match_All_3
 2018 - 8               MATCH             MATCH                   MISMATCH    MISMATCH
2019 - 33               MATCH             MATCH                   MISMATCH    MISMATCH
 2020 - 8            MISMATCH          MISMATCH                   MISMATCH    MISMATCH
2021 - 28            MISMATCH             MATCH                   MISMATCH    MISMATCH
2021 - 56               MATCH          MISMATCH                   MISMATCH    MISMATCH
2023 - 58               MATCH          MISMATCH                   MISMATCH    MISMATCH
2024 - 99            MISMATCH          MISMATCH                   MISMATCH    MISMATCH


In [5]:
# Multi-label metrics per dimension
if len(comparison_df) == 0:
    raise ValueError("No rows available for evaluation.")

all_metrics_rows = []

for dim in DIMENSIONS:
    y_true_labels = comparison_df[f"GT_{dim}"].tolist()
    y_pred_labels = comparison_df[f"Agent_{dim}"].tolist()

    mlb = MultiLabelBinarizer(classes=LABEL_SPACES[dim])
    mlb.fit([LABEL_SPACES[dim]])

    y_true = mlb.transform(y_true_labels)
    y_pred = mlb.transform(y_pred_labels)

    subset_acc = accuracy_score(y_true, y_pred)
    precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    all_metrics_rows.append({
        "dimension": dim,
        "samples": len(y_true_labels),
        "subset_accuracy": subset_acc,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    })

    print("=" * 90)
    print(f"DIMENSION: {dim}")
    print("=" * 90)
    print(f"subset_accuracy........ {subset_acc:.4f}")
    print(f"precision_micro....... {precision_micro:.4f}")
    print(f"recall_micro.......... {recall_micro:.4f}")
    print(f"f1_micro.............. {f1_micro:.4f}")
    print(f"precision_macro....... {precision_macro:.4f}")
    print(f"recall_macro.......... {recall_macro:.4f}")
    print(f"f1_macro.............. {f1_macro:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=mlb.classes_,
        zero_division=0,
        output_dict=True
    )
    report_df = pd.DataFrame(report).T
    print("\nPer-label report:")
    print(report_df.to_string())

metrics_df = pd.DataFrame(all_metrics_rows)
print("\nSUMMARY TABLE")
print(metrics_df.to_string(index=False))

DIMENSION: data_category
subset_accuracy........ 0.5714
precision_micro....... 0.7000
recall_micro.......... 0.8750
f1_micro.............. 0.7778
precision_macro....... 0.5208
recall_macro.......... 0.6875
f1_macro.............. 0.5536

Per-label report:
                    precision    recall  f1-score  support
UGC                  0.750000  1.000000  0.857143      3.0
Device               0.000000  0.000000  0.000000      0.0
Transaction          0.333333  1.000000  0.500000      1.0
Survey/Statistical   1.000000  0.750000  0.857143      4.0
micro avg            0.700000  0.875000  0.777778      8.0
macro avg            0.520833  0.687500  0.553571      8.0
weighted avg         0.822917  0.875000  0.812500      8.0
samples avg          0.690476  0.857143  0.738095      8.0
DIMENSION: data_format
subset_accuracy........ 0.4286
precision_micro....... 0.7143
recall_micro.......... 0.5556
f1_micro.............. 0.6250
precision_macro....... 0.3333
recall_macro.......... 0.2778
f1_macro..

# Detailed mismatch diagnostics

In [6]:
# Detailed mismatch diagnostics
def labels_to_text(labels):
    return "; ".join(labels) if labels else "(none)"

detailed = comparison_df.copy()
for dim in DIMENSIONS:
    detailed[f"GT_{dim}"] = detailed[f"GT_{dim}"].apply(labels_to_text)
    detailed[f"Agent_{dim}"] = detailed[f"Agent_{dim}"].apply(labels_to_text)

print("DETAILED COMPARISON: data_understanding\n")
show_cols = [
    "Paper ID",
    "GT_data_category", "Agent_data_category", "Match_data_category",
    "GT_data_format", "Agent_data_format", "Match_data_format",
    "GT_data_characteristics", "Agent_data_characteristics", "Match_data_characteristics",
    "Match_All_3",
]
print(detailed[show_cols].to_string(index=False))

all_match_count = (comparison_df["Match_All_3"] == "MATCH").sum()
total = len(comparison_df)
print(f"\nOverall exact agreement across all 3 dimensions: {all_match_count}/{total} ({(100 * all_match_count / total):.1f}%)")

wrong_paper_ids = comparison_df[comparison_df["Match_All_3"] == "MISMATCH"]["Paper ID"].tolist()
correct_paper_ids = comparison_df[comparison_df["Match_All_3"] == "MATCH"]["Paper ID"].tolist()

print(f"\nIncorrect papers (any mismatch): {len(wrong_paper_ids)}")
if wrong_paper_ids:
    print(", ".join(wrong_paper_ids))

print(f"\nFully correct papers (all 3 match): {len(correct_paper_ids)}")
if correct_paper_ids:
    print(", ".join(correct_paper_ids))

DETAILED COMPARISON: data_understanding

 Paper ID   GT_data_category                  Agent_data_category Match_data_category           GT_data_format           Agent_data_format Match_data_format      GT_data_characteristics Agent_data_characteristics Match_data_characteristics Match_All_3
 2018 - 8 Survey/Statistical                   Survey/Statistical               MATCH               Structured                  Structured             MATCH   Spatial; Temporal; Textual                   Temporal                   MISMATCH    MISMATCH
2019 - 33                UGC                                  UGC               MATCH               Structured                  Structured             MATCH            Temporal; Textual                    Textual                   MISMATCH    MISMATCH
 2020 - 8 Survey/Statistical Survey/Statistical; Transaction; UGC            MISMATCH Structured; Unstructured Semi-Structured; Structured          MISMATCH  Networked; Spatial; Textual         Networked

In [7]:
# Label-level mismatch attribution: identify highest failure-contributing labels
if len(comparison_df) == 0:
    raise ValueError("comparison_df is empty. Run previous evaluation cells first.")

label_error_rows = []
n_samples = len(comparison_df)

for dim in DIMENSIONS:
    for label in LABEL_SPACES[dim]:
        tp = fp = fn = tn = 0
        mismatch_papers = []

        for _, row in comparison_df.iterrows():
            paper_id = row["Paper ID"]
            gt_set = set(row[f"GT_{dim}"])
            pred_set = set(row[f"Agent_{dim}"])

            gt_has = label in gt_set
            pred_has = label in pred_set

            if gt_has and pred_has:
                tp += 1
            elif (not gt_has) and pred_has:
                fp += 1
                mismatch_papers.append(paper_id)
            elif gt_has and (not pred_has):
                fn += 1
                mismatch_papers.append(paper_id)
            else:
                tn += 1

        error_count = fp + fn
        support = tp + fn
        predicted_positive = tp + fp
        mismatch_rate = (error_count / n_samples) if n_samples else 0.0

        label_error_rows.append({
            "dimension": dim,
            "label": label,
            "support_gt": support,
            "predicted_positive": predicted_positive,
            "false_positives": fp,
            "false_negatives": fn,
            "total_mismatch_count": error_count,
            "mismatch_rate": mismatch_rate,
            "mismatch_papers": mismatch_papers,
        })

label_error_df = pd.DataFrame(label_error_rows)
total_label_mismatches = int(label_error_df["total_mismatch_count"].sum())

if total_label_mismatches > 0:
    label_error_df["contribution_pct"] = (
        label_error_df["total_mismatch_count"] / total_label_mismatches * 100.0
    )
else:
    label_error_df["contribution_pct"] = 0.0

label_error_df = label_error_df.sort_values(
    by=["total_mismatch_count", "false_negatives", "false_positives", "dimension", "label"],
    ascending=[False, False, False, True, True]
).reset_index(drop=True)

print("=" * 100)
print("LABEL-LEVEL MISMATCH CONTRIBUTION RANKING (HIGH TO LOW)")
print("=" * 100)
print(f"Samples evaluated: {n_samples}")
print(f"Total label mismatches across all dimensions: {total_label_mismatches}")

if len(label_error_df) and label_error_df.iloc[0]["total_mismatch_count"] > 0:
    top = label_error_df.iloc[0]
    print("\nTop failure-contributing label:")
    print(
        f"- Dimension: {top['dimension']} | Label: {top['label']} | "
        f"Mismatches: {int(top['total_mismatch_count'])} "
        f"(FN={int(top['false_negatives'])}, FP={int(top['false_positives'])}) | "
        f"Contribution: {top['contribution_pct']:.1f}%"
    )
else:
    print("\nNo label-level mismatches found.")

display_cols = [
    "dimension",
    "label",
    "support_gt",
    "predicted_positive",
    "false_positives",
    "false_negatives",
    "total_mismatch_count",
    "mismatch_rate",
    "contribution_pct",
]
print("\nFull ranking table:")
print(label_error_df[display_cols].to_string(index=False))

print("\nPapers impacted by each mismatching label:")
for _, r in label_error_df.iterrows():
    if r["total_mismatch_count"] == 0:
        continue
    papers = ", ".join(r["mismatch_papers"]) if r["mismatch_papers"] else "-"
    print(
        f"- {r['dimension']} -> {r['label']}: "
        f"{int(r['total_mismatch_count'])} mismatches | Papers: {papers}"
    )

LABEL-LEVEL MISMATCH CONTRIBUTION RANKING (HIGH TO LOW)
Samples evaluated: 7
Total label mismatches across all dimensions: 21

Top failure-contributing label:
- Dimension: data_characteristics | Label: Spatial | Mismatches: 4 (FN=4, FP=0) | Contribution: 19.0%

Full ranking table:
           dimension              label  support_gt  predicted_positive  false_positives  false_negatives  total_mismatch_count  mismatch_rate  contribution_pct
data_characteristics            Spatial           5                   1                0                4                     4       0.571429         19.047619
data_characteristics            Textual           5                   2                0                3                     3       0.428571         14.285714
         data_format       Unstructured           3                   0                0                3                     3       0.428571         14.285714
data_characteristics           Temporal           5                   3   

In [16]:
# Single-paper testing harness for data_understanding_node
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Set the paper you want to test (case-insensitive)
TEST_PAPER_ID = "2021 - 28"

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

paper_id_norm = normalize_paper_id(TEST_PAPER_ID)
if not paper_id_norm:
    raise ValueError("Please provide a valid TEST_PAPER_ID.")

benchmark_row = benchmark_df_clean[
    benchmark_df_clean["Paper ID"].astype(str).str.strip().str.lower() == paper_id_norm
]
if benchmark_row.empty:
    raise ValueError(f"Paper ID not found in benchmark: {TEST_PAPER_ID}")

gatekeeper_value = normalize_text(benchmark_row.iloc[0].get("Gatekeeper"))
if (gatekeeper_value or "").lower() != "include":
    raise ValueError(
        f"Paper {TEST_PAPER_ID} has Gatekeeper={gatekeeper_value}; single-paper evaluation expects Include."
    )

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

state = DSRPState(
    paper_id=paper_id_norm,
    gatekeeper={},
    dsrp_outputs={},
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_PATH),
    embedding_model=EMBEDDING_MODEL,
    llm_model=LLM_MODEL,
 )

try:
    out = data_understanding_node(state)
    outputs = get_nested(out, "dsrp_outputs", default={})
    pred_obj = outputs.get("data_understanding", {})
    pre_audit = outputs.get("data_understanding_pre_audit", {})
    classifier_outputs = pre_audit.get("classification_outputs", {})

    print("=" * 90)
    print(f"SINGLE-PAPER TEST: {TEST_PAPER_ID}")
    print(f"Model: {LLM_MODEL}")
    print("=" * 90)

    print("Pre-audit classifier outputs:")
    for dim_key in ["data_category", "data_format", "data_characteristics"]:
        print(f"  {dim_key}: {classifier_outputs.get(dim_key, {})}")

    overall_match_flags = []
    for dim in DIMENSIONS:
        gt_labels = sorted(set(parse_multilabel(benchmark_row.iloc[0].get(dim), dim)))
        pred_labels = sorted(set(parse_multilabel(pred_obj.get(dim), dim)))
        match = gt_labels == pred_labels
        overall_match_flags.append(match)

        print(f"\n{dim}")
        print(f"  GT    : {gt_labels if gt_labels else ['(none)']}")
        print(f"  Agent : {pred_labels if pred_labels else ['(none)']}")
        print(f"  Match : {'MATCH' if match else 'MISMATCH'}")

        fn_labels = sorted(set(gt_labels) - set(pred_labels))
        fp_labels = sorted(set(pred_labels) - set(gt_labels))
        if fn_labels:
            print(f"  Missing labels (FN): {fn_labels}")
        if fp_labels:
            print(f"  Extra labels (FP): {fp_labels}")

    print("\n" + "-" * 90)
    print(f"Overall all-3-dimensions match: {'MATCH' if all(overall_match_flags) else 'MISMATCH'}")

except Exception as e:
    print(f"Error while testing {TEST_PAPER_ID}: {e}")
finally:
    os.chdir(original_dir)

SINGLE-PAPER TEST: 2021 - 28
Model: gpt-4o-mini
Pre-audit classifier outputs:
  data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is sourced from official statistical organizations such as the World Bank, OECD, IMF, and WHO, which qualifies it as Survey/Statistical. Additionally, the mention of Datastream indicates the use of a database that compiles financial and economic data, further supporting the classification.', 'bibliography': [{'id': 2, 'page': '4', 'section': '3.4. Country-level variables', 'direct_quote': 'The data is compiled from such sources as the World Bank, OECD national accounts, the International Monetary Fund, the World Tourism Organization, and the World Health Organization (WHO).'}, {'id': 1, 'page': '3', 'section': '3.1. Sample of stocks', 'direct_quote': 'We retrieve a global selection of travel and leisure companies with Datastream, which provides 2881 equities classified to the travel and leisure ind

## Re-run Data Understanding Node for Incorrect Papers
After updating prompts in `prompts/dsrp/data_understanding/`, run the next cell to re-test only the incorrect papers.

In [52]:
## Per-Label Data Characteristics Classifier Outputs (Pre-Audit Detailed)
# Display individual responses from each characteristic label classifier

if 'pre_audit' in dir() and pre_audit:
    print("=" * 110)
    print("PER-LABEL DATA CHARACTERISTICS CLASSIFIER OUTPUTS (Before Auditor)")
    print("=" * 110)
    
    characteristics_labels = ["Temporal", "Spatial", "Textual", "Visual", "Networked"]
    
    # Extract evidence structure
    evidence_struct = pre_audit.get("combined_classification", {}).get("evidence", {}).get("data_characteristics", {})
    per_label_classifiers = evidence_struct.get("per_label_classifiers", {})
    by_label_evidence = evidence_struct.get("by_label", {})
    
    if per_label_classifiers:
        print("\nDETAILED LABEL-BY-LABEL CLASSIFICATION RESULTS:\n")
        for label in characteristics_labels:
            label_output = per_label_classifiers.get(label, {})
            
            if label_output:
                is_present = label_output.get("is_present", False)
                confidence = label_output.get("confidence", 0.0)
                reasoning = label_output.get("reasoning_explanation", "")
                bibliography = label_output.get("bibliography", [])
                
                # Visual indicator
                status = "✓ SELECTED" if is_present else "✗ REJECTED"
                print(f"{status:15} | {label:12} | Confidence: {confidence:.2f}")
                
                # Full reasoning without truncation
                if reasoning:
                    print(f"{'':15}   Reasoning:")
                    print(reasoning)
                else:
                    print(f"{'':15}   Reasoning: (none)")
                
                # Full bibliography metadata without truncation
                if bibliography:
                    print(f"{'':15}   Evidence: {len(bibliography)} citation(s)")
                    for i, bib in enumerate(bibliography, 1):
                        page = bib.get("page", "?")
                        section = bib.get("section", "")
                        quote = bib.get("direct_quote", "")
                        print(f"{'':15}     [{i}] p.{page} | {section}")
                        print(f"{'':15}         Quote:")
                        print(quote if quote else f"{'':15}         (no direct quote provided)")
                else:
                    print(f"{'':15}   Evidence: None")
                print()
            else:
                print(f"✗ REJECTED     | {label:12} | (No output)")
                print()
        
        # Show auditor changes
        print("\n" + "=" * 110)
        print("CHARACTERISTICS AUDITOR DECISIONS:")
        print("=" * 110)
        
        # Get the final audited result
        final_chars = []
        if 'pred_obj' in dir() and pred_obj:
            final_chars = pred_obj.get("data_characteristics", [])
            auditor_commentary = pred_obj.get("audit_commentary", "")
            
            print(f"\nFinal Selected: {final_chars}")
            if auditor_commentary:
                print("Auditor Commentary:")
                print(auditor_commentary)
        
        # Show which labels were kept/rejected by auditor
        print("\nLabel Audit Results (Full):")
        class_output = pre_audit.get("classification_outputs", {}).get("data_characteristics", {})
        pre_audit_chars = class_output.get("data_characteristics", [])
        print(f"  Pre-audit: {pre_audit_chars}")
        print(f"  Post-audit: {final_chars if 'pred_obj' in dir() else 'N/A'}")
    else:
        print("Per-label classifier outputs not found in pre_audit structure.")
        print("This may be because the node hasn't been run with the updated version yet.")
        print("\nAvailable pre_audit keys:", list(pre_audit.keys()))
else:
    print("Pre-audit data not available. Run the single-paper test cell first.")

PER-LABEL DATA CHARACTERISTICS CLASSIFIER OUTPUTS (Before Auditor)

DETAILED LABEL-BY-LABEL CLASSIFICATION RESULTS:

✓ SELECTED      | Temporal     | Confidence: 1.00
                  Reasoning:
The analysis includes the growth rate of the cumulative number of confirmed COVID-19 cases reported per week in each country, indicating a temporal dimension in the evaluation of stock price movements.
                  Evidence: 1 citation(s)
                    [1] p.4 | 4. Methods
                        Quote:
Our model assumes that the key factor explaining cross-country differences is the Δ COVID-19 variable representing the growth rate of the cumulative number of confirmed cases reported per week in each country.

✓ SELECTED      | Spatial      | Confidence: 1.00
                  Reasoning:
The paper explicitly analyzes country-level characteristics and includes country-specific factors as part of the data analysis, indicating geographic use in modeling and comparison.
                

### Debug: Per-Label Characteristics Outputs
The cell above shows detailed responses from each data characteristics label classifier (Temporal, Spatial, Textual, Visual, Networked) **before** the characteristics auditor filters them.

**For each label, you'll see:**
- Selection status (✓ SELECTED or ✗ REJECTED by that label's classifier)
- Confidence score (0.0 - 1.0)
- Reasoning explanation from the classifier
- Supporting bibliography with direct quotes from the paper

**Then after all label classifiers**, the characteristics-level auditor reviews all 5 label outputs and decides the final set. You'll see what changed:
- Pre-audit: labels selected by individual classifiers
- Post-audit: final labels after auditor validation

This setup lets you debug:
- Why a specific label is/isn't present
- What evidence each label found or missed
- How the auditor changed the results

In [24]:
for paper_id in wrong_paper_ids:
  
    print(paper_id)

2018 - 8
2019 - 33
2020 - 8
2021 - 28
2021 - 56
2023 - 58
2024 - 99


In [8]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a statistical source. The evidence indicates that the data was collected through systematic sampling and includes responses from a questionnaire, confirming its classification as Survey/Statistical.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset collected from a survey, which includes structured variabl

**Iteration 2** To check stochastic variance. 

In [10]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a model with a nonnegative dependent variable and mentions the inclusion of sociodemographic and expenditure data, which s

**Iteration 3** To check stochastic variance. *After Prompt Changes*

In [14]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly described as coming from a survey conducted by the Ministry of Tourism of Uruguay, which aligns with the Survey/Statistical category.', 'bibliography': [{'id': 1, 'page': '2', 'section': 'Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': 'Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).'}, {'id': 3, 'page': '2', 'section': 'Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 4, 'page': '2', 'section': 'Dataset descripti

**Iteration 4**

In [15]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly described as coming from a survey conducted by the Ministry of Tourism of Uruguay, which aligns with the Survey/Statistical category.', 'bibliography': [{'id': 1, 'page': '2', 'section': 'Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': 'Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 5, 'page': '2', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation'

**Iteration 5**

In [18]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 4, 'page': '2', 'section': '2. Dataset desc

**Iteration 6** *After undo changes made for 2021 - 28, yet keeping the earlier changes made for 2018 -8*.

In [19]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a statistical source. The evidence clearly indicates that the data was collected through a systematic sampling method and includes responses from a questionnaire.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote'

**Iteration 7** *re-testing above results without any changes*. Iteration 6-7 results shall be compared with 1-2 before any prompt changes. 

In [20]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the us

**Iteration 8**

In [21]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data from the o /uniFB03 cial survey conducted in Uruguay by Ministry of Tourism (Ministerio de Turismo (2017)).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a questionnaire that includes sociodemographic and satisfaction inquiries, which suggests that the data is org

**Iteration 9** After miminising data characteristics prompts.

In [22]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data. The evidence provided directly mentions the collection of data through a survey and includes details about the questionnaire used.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).

**Iteration 10**

In [23]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset that includes structured variables such as sociodemographic information and expenditure data, which are analyzed using stat

**Iteration 11** Removing rules completely. 

In [24]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset that includes structured variables such as sociodemographic information and expenditure data, which are analyzed using stat

**Iteration 12** After making data characteristics prompts less verbose, instead of complete removal. 

In [25]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a questionnaire that includes sociodemographic and satisfaction inquiries, which suggests that the data is organized in a 

**Iteration 13** Bifurcating data characteristics prompt into six separate prompts.

In [27]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 6, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data from the official survey conducted in Uruguay by Ministry of Tourism (Ministerio de Turismo (2017)).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset collected from a survey, which includes structured variables such as sociodemographic information and expenditure data

**Iteration 14** Reduce LLM calls by merging retriever+classifiers into single call. Additionally, using the same rules that we defined earlier duing single data characteristics call. Stricter rules for label selecton!.

In [30]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a statistical source. The evidence clearly indicates that the data comes from a systematic sampling of tourists and includes inquiries from a questionnaire.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Dat

**Iteration 15** Less strict rules, rather rules re-written again grounding in the older monolithic prompt. Added a data characteristics level auditor. 

In [33]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a dataset collected from a survey, which includes structured variables such as sociodemographic information and expenditur

**Iteration 16** Performance re-test before refining prompts again. 

In [34]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation':

**Iteration 17**

In [8]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data. The references to systematic sampling and the questionnaire further support this classification.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a questionnaire that includes sociodemographic and satisfaction inquiries, which suggests that the data is organ

**Iteration 18** After making visual classifier prompt more direct and clear.  

In [10]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset that includes structured variables such as sociodemographic information and expenditure data, which are analyzed using stat

**Iteration 19** After fine-tuning prompt for textual. I want to see if its creating problems to earlier predictions. 

In [21]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data from the o /uniFB03 cial survey conducted in Uruguay by Ministry of Tourism (Ministerio de Turismo (2017)).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset collected from a survey, which includes structured variables such as sociodemographic data and expenditure metr

**Iteration 20** After fixing visual prompt. 

In [25]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a dataset that includes sociodemographic and expenditure variables, which are analyzed using statistical and econometric m

**Iteration 21** After fixing temporal prompt. 

In [46]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")
    

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '2', 'section': 'A B S T R A C T', 'direct_quote': 'using data from the o /uniFB03 cial survey conducted in Uruguay by Ministry of Tourism (Ministerio de Turismo (2017)).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a questionnaire that includes sociodemographic and satisfaction inquiries, which suggests that the data is org

**Iteration 22** re-testing

In [47]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data from the o /uniFB03 cial survey conducted in Uruguay by Ministry of Tourism (Ministerio de Turismo (2017)).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes the use of a questionnaire that includes sociodemographic and satisfaction inquiries, which suggests that the data is org

**Iteration 23** After easing temporal logics. 

In [48]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 6, 'page': '2', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 1.0, 'reasoning_explana

**Iteration 24** re-testing

In [49]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 4, 'page': '2', 'section': '2. Datase

**Iteration 25** re-testing

In [50]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a statistical source. The evidence clearly indicates that the data was collected through a systematic sampling method and includes responses from a questionnaire.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 2, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The sample was obtained through a systematic sampling on the expected arrivals of the cruisers; in each of them, a simple random sample of tourists is taken.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote'

**Iteration 26** re-testing after environment restart. 

In [10]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which includes sociodemographic and satisfaction inquiries. This aligns with the definition of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_form

**Iteration 27** re-testing. LLM Temprature Set 0.5 in Backend. 

In [11]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly stated to be collected from a survey conducted by the Ministry of Tourism of Uruguay, which qualifies it as Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 4, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'The questionnaire includes sociodemographic, context and satisfaction inquiries.'}, {'id': 3, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Data is open and available at Ministerio de Turismo (2017).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a data

**Iteration 28** re-testing. LLM Temp 0.5.

In [15]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in ["data_category", "data_format", "data_characteristics"]:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (INCORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running data_understanding node for 2018 - 8 with gpt-4o-mini...
  Pre-audit classifier outputs:
    data_category: {'data_category': ['Survey/Statistical'], 'confidence': 1.0, 'reasoning_explanation': 'The data is explicitly sourced from a survey conducted by the Ministry of Tourism of Uruguay, which is a clear indication of Survey/Statistical data.', 'bibliography': [{'id': 1, 'page': '2', 'section': '2. Dataset description', 'direct_quote': 'Individual data came from the 2016 -2017 season survey of cruisers, collected by the Ministry of Tourism of Uruguay.'}, {'id': 5, 'page': '1', 'section': 'A B S T R A C T', 'direct_quote': 'using data of the 2016 -2017 cruise season survey (collected by the Ministry of Tourism of Uruguay).'}]}
    data_format: {'data_format': ['Structured'], 'confidence': 0.9, 'reasoning_explanation': 'The study describes a dataset that includes structured variables such as sociodemographic information and expenditure data, which are analyzed using statistica

**Note** 

The results have stabilised now, there is minor variation I think this will remain given the Stochastic Nature of the models. 

I also used **LLM Temparature** as **0.5**. 

Later may be adding higher models could also increase accuracy. Models like thinking models or models from other providers like Anthropic. 

## Compare Label Contributions: Before vs After Prompt Changes

In [ ]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState
import json

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Test paper
TEST_PAPER_ID = "2021 - 28"
LLM_MODEL = "gpt-4o-mini"

paper_id_norm = normalize_paper_id(TEST_PAPER_ID)
if not paper_id_norm:
    raise ValueError("Please provide a valid TEST_PAPER_ID.")

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

state = DSRPState(
    paper_id=paper_id_norm,
    gatekeeper={},
    dsrp_outputs={},
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_PATH),
    embedding_model=EMBEDDING_MODEL,
    llm_model=LLM_MODEL,
)

try:
    out = data_understanding_node(state)
    pre_audit = get_nested(out, "dsrp_outputs.data_understanding_pre_audit", default={})
    
    print("=" * 100)
    print(f"DATA CHARACTERISTICS LABEL CLASSIFIER OUTPUTS (Before Auditor)")
    print(f"Paper: {TEST_PAPER_ID} | Model: {LLM_MODEL}")
    print("=" * 100)
    
    # Display outputs from each individual label classifier
    label_classifiers = ["Temporal", "Spatial", "Textual", "Visual", "Networked"]
    characteristics_by_label = pre_audit.get("classification_outputs", {}).get("characteristics_by_label", {})
    
    # If structure is different, try to extract it directly
    if not characteristics_by_label:
        # Try alternate structure
        classification_outputs = pre_audit.get("classification_outputs", {})
        if "data_characteristics" in classification_outputs and isinstance(classification_outputs.get("data_characteristics"), dict):
            # Single combined output, need to show component details
            print("\nFINAL DATA CHARACTERISTICS OUTPUT:")
            print(json.dumps(classification_outputs["data_characteristics"], indent=2))
        else:
            characteristics_by_label = classification_outputs
    
    if characteristics_by_label:
        for label_name in label_classifiers:
            label_output = characteristics_by_label.get(label_name, {})
            print(f"\n{'-' * 100}")
            print(f"CLASSIFIER OUTPUT: {label_name}")
            print(f"{'-' * 100}")
            print(f"  is_present: {label_output.get('is_present', 'N/A')}")
            print(f"  confidence: {label_output.get('confidence', 'N/A')}")
            print(f"  reasoning:  {label_output.get('reasoning_explanation', 'N/A')[:150]}...")
            
            biblio = label_output.get("bibliography", [])
            print(f"  bibliography count: {len(biblio)}")
            if biblio:
                for bib in biblio[:2]:  # Show first 2 citations
                    print(f"    - [ID {bib.get('id')}] {bib.get('section', '')} | {bib.get('direct_quote', '')[:80]}...")
    
    print(f"\n{'=' * 100}")
    print("AUDITOR INPUT (all label outputs):")
    print(f"{'=' * 100}")
    auditor_input_section = pre_audit.get("classification_outputs", {})
    if "characteristics_by_label" in auditor_input_section or len([k for k in label_classifiers if k in auditor_input_section]) > 0:
        print(json.dumps(auditor_input_section, indent=2, default=str)[:2000])
    else:
        print("(auditor input structure differs from expected)")
    
    print(f"\n{'=' * 100}")
    print("FINAL AUDITED RESULT (After Auditor Processing):")
    print(f"{'=' * 100}")
    final_output = out.get("dsrp_outputs", {}).get("data_understanding", {})
    print(f"  data_characteristics: {final_output.get('data_characteristics', [])}")
    print(f"  confidence: {final_output.get('confidence', 'N/A')}")
    print(f"  reasoning: {final_output.get('reasoning_explanation', 'N/A')[:200]}...")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    os.chdir(original_dir)# Build updated comparison dataframe from retest results (incorrect + correct papers)
# Then run label-level analysis and compare against Cell 8 baseline

if 'retest_df' not in dir() or len(retest_df) == 0:
    raise ValueError("retest_df from Cell 12 is empty. Please run Cell 12 (incorrect papers retest) first.")

print("=" * 100)
print("PREPARE UPDATED PREDICTIONS FOR ALL PAPERS")
print("=" * 100)

# Collect results from both retests (incorrect and correct papers)
all_retest_results = []

# Add results from incorrect papers retest (Cell 12 output)
for _, row in retest_df.iterrows():
    if "Error" in row and pd.notna(row["Error"]):
        print(f"Skipping {row['Paper ID']} due to error in retest")
        continue
    
    retest_row = {
        "Paper ID": row["Paper ID"],
        "source": "incorrect_retest",
    }
    for dim in DIMENSIONS:
        if f"New_Agent_{dim}" in row:
            retest_row[f"New_Agent_{dim}"] = row[f"New_Agent_{dim}"]
        elif f"Agent_{dim}" in row:  # Fallback if column naming varies
            retest_row[f"New_Agent_{dim}"] = row[f"Agent_{dim}"]
    all_retest_results.append(retest_row)

print(f"Collected retest results for {len(all_retest_results)} incorrect papers")

# Now build updated comparison_df by merging retest predictions
comparison_new = comparison_df.copy()

# Update predictions for papers that were retested
for idx, retest_row in retest_df.iterrows():
    if "Error" in retest_row and pd.notna(retest_row["Error"]):
        continue
    
    paper_id = retest_row["Paper ID"]
    mask_idx = comparison_new[comparison_new["Paper ID"] == paper_id].index
    
    if len(mask_idx) == 0:
        continue
    
    row_idx = mask_idx[0]  # Get the actual integer index
    
    for dim in DIMENSIONS:
        if f"New_Agent_{dim}" in retest_row:
            # Extract the value properly from the Series
            new_pred = retest_row[f"New_Agent_{dim}"]
            comparison_new.at[row_idx, f"Agent_{dim}"] = new_pred
            
            # Recalculate match flag
            gt_val = comparison_new.at[row_idx, f"GT_{dim}"]
            pred_val = comparison_new.at[row_idx, f"Agent_{dim}"]
            match = "MATCH" if gt_val == pred_val else "MISMATCH"
            comparison_new.at[row_idx, f"Match_{dim}"] = match

# Recalculate all-3 match
def check_all_match(row):
    return all([
        row.get(f"Match_{dim}") == "MATCH" 
        for dim in DIMENSIONS
    ])

comparison_new["Match_All_3"] = comparison_new.apply(
    lambda row: "MATCH" if check_all_match(row) else "MISMATCH",
    axis=1
)

print(f"\nUpdated comparison_df dimensions: {comparison_new.shape}")
print(f"Overall exact match (after retest): {(comparison_new['Match_All_3'] == 'MATCH').sum()}/{len(comparison_new)}")

# ============================================================================
# RUN LABEL-LEVEL ERROR ANALYSIS ON UPDATED PREDICTIONS (same as Cell 8)
# ============================================================================
print("\n" + "=" * 100)
print("LABEL-LEVEL ERROR ANALYSIS: AFTER PROMPT CHANGES")
print("=" * 100)

label_error_rows_new = []
n_samples_new = len(comparison_new)

for dim in DIMENSIONS:
    for label in LABEL_SPACES[dim]:
        tp = fp = fn = tn = 0
        mismatch_papers = []

        for _, row in comparison_new.iterrows():
            paper_id = row["Paper ID"]
            gt_set = set(row[f"GT_{dim}"])
            pred_set = set(row[f"Agent_{dim}"])

            gt_has = label in gt_set
            pred_has = label in pred_set

            if gt_has and pred_has:
                tp += 1
            elif (not gt_has) and pred_has:
                fp += 1
                mismatch_papers.append(paper_id)
            elif gt_has and (not pred_has):
                fn += 1
                mismatch_papers.append(paper_id)
            else:
                tn += 1

        error_count = fp + fn
        support = tp + fn
        predicted_positive = tp + fp
        mismatch_rate = (error_count / n_samples_new) if n_samples_new else 0.0

        label_error_rows_new.append({
            "dimension": dim,
            "label": label,
            "support_gt": support,
            "predicted_positive": predicted_positive,
            "false_positives": fp,
            "false_negatives": fn,
            "total_mismatch_count": error_count,
            "mismatch_rate": mismatch_rate,
            "mismatch_papers": mismatch_papers,
        })

label_error_df_new = pd.DataFrame(label_error_rows_new)
total_label_mismatches_new = int(label_error_df_new["total_mismatch_count"].sum())

if total_label_mismatches_new > 0:
    label_error_df_new["contribution_pct"] = (
        label_error_df_new["total_mismatch_count"] / total_label_mismatches_new * 100.0
    )
else:
    label_error_df_new["contribution_pct"] = 0.0

label_error_df_new = label_error_df_new.sort_values(
    by=["total_mismatch_count", "false_negatives", "false_positives", "dimension", "label"],
    ascending=[False, False, False, True, True]
).reset_index(drop=True)

print(f"\nTotal label mismatches after prompt changes: {total_label_mismatches_new}")
print("\nLabel-level error ranking (after changes):")
display_cols = [
    "dimension",
    "label",
    "support_gt",
    "false_negatives",
    "false_positives",
    "total_mismatch_count",
    "contribution_pct",
]
print(label_error_df_new[display_cols].to_string(index=False))

# ============================================================================
# COMPARE BASELINE vs UPDATED METRICS
# ============================================================================
print("\n" + "=" * 100)
print("BEFORE vs AFTER COMPARISON")
print("=" * 100)

print(f"\nBASELINE (Cell 8):          {total_label_mismatches} total label mismatches")
print(f"AFTER CHANGES (Now):        {total_label_mismatches_new} total label mismatches")
print(f"CHANGE:                     {total_label_mismatches_new - total_label_mismatches:+d} "
      f"({((total_label_mismatches_new / total_label_mismatches - 1) * 100):+.1f}%)")

# Build comparison table at label level
comparison_table = []
for _, row_new in label_error_df_new.iterrows():
    dim = row_new["dimension"]
    label = row_new["label"]
    
    # Find baseline row
    baseline_row = label_error_df[
        (label_error_df["dimension"] == dim) & 
        (label_error_df["label"] == label)
    ]
    
    if baseline_row.empty:
        baseline_errors = 0
        baseline_fn = 0
        baseline_fp = 0
        baseline_contrib = 0
    else:
        baseline_row = baseline_row.iloc[0]
        baseline_errors = int(baseline_row["total_mismatch_count"])
        baseline_fn = int(baseline_row["false_negatives"])
        baseline_fp = int(baseline_row["false_positives"])
        baseline_contrib = baseline_row["contribution_pct"]
    
    new_errors = int(row_new["total_mismatch_count"])
    new_fn = int(row_new["false_negatives"])
    new_fp = int(row_new["false_positives"])
    new_contrib = row_new["contribution_pct"]
    
    error_change = new_errors - baseline_errors
    fn_change = new_fn - baseline_fn
    fp_change = new_fp - baseline_fp
    contrib_change = new_contrib - baseline_contrib
    
    comparison_table.append({
        "dimension": dim,
        "label": label,
        "baseline_errors": baseline_errors,
        "new_errors": new_errors,
        "error_Δ": error_change,
        "baseline_FN": baseline_fn,
        "new_FN": new_fn,
        "FN_Δ": fn_change,
        "baseline_FP": baseline_fp,
        "new_FP": new_fp,
        "FP_Δ": fp_change,
        "baseline_contrib%": baseline_contrib,
        "new_contrib%": new_contrib,
        "contrib_Δ": contrib_change,
    })

comparison_table_df = pd.DataFrame(comparison_table)

# Sort by absolute error change (highest improvements/regressions first)
comparison_table_df["abs_error_change"] = comparison_table_df["error_Δ"].abs()
comparison_table_df = comparison_table_df.sort_values(
    by=["abs_error_change", "error_Δ"],
    ascending=[False, False]
).reset_index(drop=True).drop(columns=["abs_error_change"])

print("\nLABEL-BY-LABEL COMPARISON (sorted by change magnitude):")
print("\nLegend: Δ = change (negative=improvement, positive=regression)")
comp_display = comparison_table_df[[
    "dimension", "label", "baseline_errors", "new_errors", "error_Δ",
    "baseline_FN", "new_FN", "FN_Δ",
    "baseline_FP", "new_FP", "FP_Δ"
]]
print(comp_display.to_string(index=False))

# Summary: which labels improved, which regressed
print("\n" + "-" * 100)
improved = comparison_table_df[comparison_table_df["error_Δ"] < 0].sort_values("error_Δ")
regressed = comparison_table_df[comparison_table_df["error_Δ"] > 0].sort_values("error_Δ", ascending=False)
neutral = comparison_table_df[comparison_table_df["error_Δ"] == 0]

print(f"\n✓ IMPROVED ({len(improved)} labels):")
if len(improved):
    for _, row in improved.iterrows():
        print(f"  - {row['dimension']} → {row['label']}: "
              f"{int(row['baseline_errors'])} → {int(row['new_errors'])} errors "
              f"(FN: {int(row['baseline_FN'])}→{int(row['new_FN'])}, "
              f"FP: {int(row['baseline_FP'])}→{int(row['new_FP'])})")
else:
    print("  (none)")

print(f"\n✗ REGRESSED ({len(regressed)} labels):")
if len(regressed):
    for _, row in regressed.iterrows():
        print(f"  - {row['dimension']} → {row['label']}: "
              f"{int(row['baseline_errors'])} → {int(row['new_errors'])} errors "
              f"(FN: {int(row['baseline_FN'])}→{int(row['new_FN'])}, "
              f"FP: {int(row['baseline_FP'])}→{int(row['new_FP'])})")
else:
    print("  (none)")

print(f"\n= UNCHANGED ({len(neutral)} labels):")
if len(neutral):
    for _, row in neutral.iterrows():
        print(f"  - {row['dimension']} → {row['label']}")
else:
    print("  (none)")

PREPARE UPDATED PREDICTIONS FOR ALL PAPERS
Collected retest results for 7 incorrect papers

Updated comparison_df dimensions: (7, 11)
Overall exact match (after retest): 3/7

LABEL-LEVEL ERROR ANALYSIS: AFTER PROMPT CHANGES

Total label mismatches after prompt changes: 11

Label-level error ranking (after changes):
           dimension              label  support_gt  false_negatives  false_positives  total_mismatch_count  contribution_pct
         data_format       Unstructured           3                2                0                     2         18.181818
data_characteristics            Spatial           4                1                1                     2         18.181818
data_characteristics            Textual           2                1                1                     2         18.181818
       data_category             Device           1                1                0                     1          9.090909
       data_category Survey/Statistical           4  

### **Notes** 

I have two ways to improve responses either keep adding rules for everything into a single classifier or divide the classifiers into three modules. I feel the second option is better as it will reduce the congnitive load in a single call, our logics will be clear. We can reduce cost by using vector_query only once, and re-use its output and retrieve relevant chunks for each sub-classifier. 

Or, may be experimenting with a better LLMs could show performance difference. But, may be that could be done in final evaluation set. 

## Re-test on Previously Correct Papers

In [ ]:
import importlib
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in correct_paper_ids:
    print(f"Re-running data_understanding node for {paper_id} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        du = get_nested(out, "dsrp_outputs.data_understanding", default={})

        row = comparison_df.loc[comparison_df["Paper ID"] == paper_id].iloc[0]
        retest_row = {"Paper ID": paper_id, "Model": LLM_MODEL}
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(row[f"GT_{dim}"]))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (PREVIOUSLY CORRECT PAPERS)")
    print(retest_df.to_string(index=False))
else:
    print("No previously correct papers to re-test.")

No previously correct papers to re-test.
